## Práctica Repaso 1
Vamos a entrenar una red neuronal en PyTorch capaz de aproximar simultáneamente dos funciones matemáticas a partir de una única variable de entrada $x \in [-100, 100]$, donde las salidas están contaminadas con ruido gaussiano:

Señal 1: $y_1 = \sin(0.05 \cdot x) + \text{ruido}$

Señal 2: $y_2 = \frac{x^2}{1000} - 2 + \text{ruido}$

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader

class SensorData(Dataset):
    def __init__(self):
        super().__init__()

        # Genero los valores de x.
        x = torch.linspace(-100, 100, 1500)

        # Genero ruido para cada función.
        noise1 = torch.randn_like(x) * 0.1
        noise2 = torch.randn_like(x) * 0.1

        # Genero los valores de y a partir de x y del ruido gaussiano.
        y1 = torch.sin(0.05 * x) + noise1
        y2 = (torch.pow(x, 2) / 1000) - 2 + noise2

        # Normalizo los valores de x a [-1, 1] y le damos forma (1500, 1).
        self.x = (x/100).unsqueeze(1)

        # Empaquetamos ambas salidas.
        self.y = torch.stack([y1, y2], dim=1)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [9]:
dataset = SensorData()
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [11]:
class NeuronalNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.entrance = torch.nn.Linear(1, 32) # OF. Usar aquí red=nn.Sequential() y en forward solo devuelvo la red.
        self.middle1 = torch.nn.Linear(32, 64)
        self.activation = torch.nn.LeakyReLU()
        self.middle2 = torch.nn.Linear(64, 32)
        self.output = torch.nn.Linear(32, 2)

    def forward(self, x):
        x = self.activation(self.entrance(x))
        x = self.activation(self.middle1(x))
        x = self.activation(self.middle2(x))
        # En regresión la última capa no lleva activación.
        x = self.output(x)
        return x


In [12]:
model = NeuronalNetwork()
loss_fn = torch.nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda")
model.to(device)

epochs = 100

for epoch in range(epochs):
    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        # Reiniciar gradientes
        optimizer.zero_grad()
        # Paso hacia delante (Predecir el modelo) - Forward
        y_pred = model(x_batch)
        # Calcular la pérdida
        loss = loss_fn(y_pred, y_batch)
        # Paso hacia detrás - Backward
        loss.backward()
        # Actualizar pesos
        optimizer.step()